In [100]:
#Feature 1
#AI-assisted (for error correction)
import pandas as pd
import numpy as np

df = pd.read_csv('rahul_transactions.csv')

#Tracking initial count and handle duplicates
initial_count = len(df)
df = df.drop_duplicates()
dropped_count = initial_count - len(df)

#Clean and Parse Amounts
df['Amount_Cleaned'] = df['Amount'].replace(r'[^\d.]', '', regex=True)
df['Amount_Numeric'] = pd.to_numeric(df['Amount_Cleaned'], errors='coerce')
unparseable_amounts = df['Amount_Numeric'].isna().sum()

df['Date_Parsed'] = pd.to_datetime(df['Date'], dayfirst=True, format='mixed', errors='coerce')
unparseable_dates = df['Date_Parsed'].isna().sum()
print(f"Parsed {initial_count} transactions across 6 months. Dropped {dropped_count} duplicates. {unparseable_amounts} unparseable amounts, {unparseable_dates} unparseable dates.")

Parsed 1328 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 0 unparseable dates.


In [101]:
# Feature 2- Vendor Extractor

# Dictionary remains as you defined it
vendor_dict = {
    'Swiggy': ['SWIGGY', 'BUNDL'], 'Zomato': ['ZOMATO'], 'Amazon': ['AMAZON', 'AMZN'],
    'Uber': ['UBER', 'ANI', 'OLA CABS'], 'Ola': ['OLA'],
    'Blinkit': ['BLINKIT', 'GROFERS', 'INNOVATIVE RETAIL', 'KIRANAKART'],
    'Zepto': ['ZEPTO'], 'Starbucks': ['STARBUCK'], 'P2P Transfer': ['UPI-', 'IMPS', 'NEFT'],
    'Cash Withdrawal': ['ATM-', 'WDL'], 'BookMyShow': ['BMS', 'BOOKMYSHOW', 'BIGTREE'],
    'Jio': ['JIO', 'RELIANCE JIO'], 'Airtel': ['AIRTEL'], 'Vodafone Idea': ['VI', 'VODAFONE'],
    'BESCOM': ['BESCOM', 'BANGALORE ELEC'], 'BigBasket': ['BIGBASKET'],
    'D-Mart': ['DMART', 'AVENUE SUPERMARTS'], 'Rapido': ['RAPIDO', 'ROPPEN'],
    'BMTC': ['BMTC', 'TUMMOC'], 'HP Petrol': ['HP PETROL'], 'BPCL': ['BPCL'],
    'Indian Oil': ['INDIAN OIL'], 'Flipkart': ['FLIPKART', 'FKART'], 'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA', 'FSN E-COMMERCE'], 'Spotify': ['SPOTIFY'], 'Netflix': ['NETFLIX'],
    'Hotstar': ['HOTSTAR'], 'Star India': ['STAR INDIA'], 'Zerodha': ['ZERODHA'],
    'Groww': ['GROWW', 'NEXTBILLION'], 'TWC India': ['TWC INDIA'],
    'Restaurants': ['TRUFFLES', 'EMPIRE', 'MEGHANA', 'DINEOUT', 'ZOMATO BANGALORE'],
    'Cafe': ['CAFE COFFEE DAY', 'COFFEE DAY', 'THIRD WAVE']
}

def extract_vendor(description, mapping):
    # Convert description to uppercase using standard string method
    desc_up = str(description).upper()
    for vendor, keywords in mapping.items():
        for keyword in keywords:
            # Standard string check (no regex)
            if keyword.upper() in desc_up:
                return vendor
    return 'Other'

# Apply the vendor extraction
df['vendor_clean'] = df['Description'].apply(lambda x: extract_vendor(x, vendor_dict))

# Verification outputs
print(f"Number of unique canonical vendors: {df['vendor_clean'].nunique()}")
print("\nTop 10 Vendors:")
print(df['vendor_clean'].value_counts().head(10))

# --- BONUS: Vendor Cleanup Audit ---
print("\n--- Vendor Cleanup Audit ---")
uncategorized = df[df['vendor_clean'] == 'Other']['Description'].unique()
print(f"Total uncategorized descriptions: {len(uncategorized)}")
print("First 5 Uncategorized Examples:")
for desc in uncategorized[:5]:
    print(f"- {desc}")

Number of unique canonical vendors: 35

Top 10 Vendors:
vendor_clean
Swiggy          223
P2P Transfer    181
Zomato          121
Uber            110
Amazon           86
Blinkit          76
Zepto            58
Ola              48
Restaurants      44
Starbucks        42
Name: count, dtype: int64

--- Vendor Cleanup Audit ---
Total uncategorized descriptions: 4
First 5 Uncategorized Examples:
- INSTAMART BANGALORE
- POS BANGALORE RESTAURANT
- POS INSTAMART
- BWSSB WATER BILL


In [102]:
# Feature 3: Category Tagger
#AI-assisted
# The mapping remains identical, but we ensure no regex is used
category_map = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce', 'Blinkit': 'Quick Commerce',
    'BigBasket': 'Groceries', 'D-Mart': 'Groceries',
    'Amazon': 'E-commerce', 'Flipkart': 'E-commerce', 'Myntra': 'E-commerce', 'Nykaa': 'E-commerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport',
    'Starbucks': 'Cafe', 'Cafe': 'Cafe',
    'Restaurants': 'Restaurants',
    'Jio': 'Utilities', 'Airtel': 'Utilities', 'Vodafone Idea': 'Utilities', 'BESCOM': 'Utilities',
    'HP Petrol': 'Fuel', 'BPCL': 'Fuel', 'Indian Oil': 'Fuel',
    'Spotify': 'Subscriptions', 'Netflix': 'Subscriptions',
    'Hotstar': 'Entertainment', 'Star India': 'Entertainment', 'BookMyShow': 'Entertainment',
    'Zerodha': 'Investments', 'Groww': 'Investments', 'TWC India': 'Investments',
    'P2P Transfer': 'Personal Transfer', 'Cash Withdrawal': 'Cash Withdrawal'
}

# Map the categories
df['category'] = df['vendor_clean'].map(category_map)
#Assign 'Miscellaneous' to unmapped entries
df['category'] = df['category'].fillna('Miscellaneous')

print("--- Category Distribution ---")
print(df['category'].value_counts())

# Audit for Bonus: Check for Miscellaneous items
misc_items = df[df['category'] == 'Miscellaneous']['vendor_clean'].unique()
print(f"\nNumber of vendors categorized as 'Miscellaneous': {len(misc_items)}")
print("First 5 'Miscellaneous' vendors:", misc_items[:5])

--- Category Distribution ---
category
Food Delivery        344
Transport            226
Personal Transfer    181
E-commerce           146
Quick Commerce       134
Cafe                  70
Restaurants           44
Miscellaneous         37
Utilities             27
Fuel                  22
Entertainment         18
Groceries             18
Cash Withdrawal       17
Investments           15
Subscriptions         11
Name: count, dtype: int64

Number of vendors categorized as 'Miscellaneous': 1
First 5 'Miscellaneous' vendors: ['Other']


In [103]:
#Feature4- Spending Overview
def clean_amount(val):
    val_str = str(val)
    # List comprehension keeps only digits and the decimal point
    clean_str = "".join([c for c in val_str if c.isdigit() or c == '.'])
    return float(clean_str) if clean_str else 0.0

# Create the numeric amount column
df['Amount_clean'] = df['Amount'].apply(clean_amount)

# Calculate Headline Financial Stats
is_credit = df['Type'].astype(str).str.contains('CR', case=False, na=False)
is_debit = df['Type'].astype(str).str.contains('DR|Debit', case=False, na=False)

total_credits = df[is_credit]['Amount_clean'].sum()
total_debits = df[is_debit]['Amount_clean'].sum()

net_savings = total_credits - total_debits
savings_rate = (net_savings / total_credits) * 100 if total_credits != 0 else 0

print("--- Executive Spending Overview ---")
print(f"Total Transaction Count : {len(df)}")
print(f"Total Credits           : ₹{total_credits:,.2f}")
print(f"Total Debits            : ₹{total_debits:,.2f}")
print(f"Net Savings             : ₹{net_savings:,.2f}")
print(f"Savings Rate            : {savings_rate:.2f}%")

print("\nTop 5 Categories by Spend:")
print(df[is_debit].groupby('category')['Amount_clean'].sum().abs().sort_values(ascending=False).head(5))

print("\nTop 5 Vendors by Spend:")
print(df[is_debit].groupby('vendor_clean')['Amount_clean'].sum().abs().sort_values(ascending=False).head(5))

--- Executive Spending Overview ---
Total Transaction Count : 1310
Total Credits           : ₹425,073.85
Total Debits            : ₹1,274,548.40
Net Savings             : ₹-849,474.55
Savings Rate            : -199.84%

Top 5 Categories by Spend:
category
Personal Transfer    384985.37665
E-commerce           377119.35008
Food Delivery        110864.01500
Restaurants           63841.41750
Quick Commerce        57764.83260
Name: Amount_clean, dtype: float64

Top 5 Vendors by Spend:
vendor_clean
P2P Transfer    384985.37665
Amazon          268025.00673
Swiggy           69861.68700
Flipkart         66152.31985
Restaurants      63841.41750
Name: Amount_clean, dtype: float64


In [104]:
#Feature5-Monthly trend analysis
import pandas as pd
import numpy as np

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

df['month'] = df['Date'].dt.month_name()

#Define the chronological order of months
month_order = ['January', 'February', 'March', 'April', 'May', 'June']
df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)

#Build the pivot table (Rows = category, Columns = months, Values = sum of amount)
month_pivot = df.pivot_table(
    values='Amount_clean',
    index='category',
    columns='month',
    aggfunc='sum'
).fillna(0)

#Compute growth using the formula
month_pivot['growth_pct'] = (month_pivot['June'] - month_pivot['January']) / \
                            month_pivot['January'].replace(0, np.nan) * 100

# Display the Matrix
print("--- Monthly Spending Matrix ---")
print(month_pivot)

#Identify Trending Categories (Biggest Growth vs Biggest Decline)
valid_growth = month_pivot.dropna(subset=['growth_pct'])

if not valid_growth.empty:
    trending_up = valid_growth['growth_pct'].idxmax()
    trending_down = valid_growth['growth_pct'].idxmin()

    print(f"\nBiggest Growth Category: {trending_up} ({valid_growth.loc[trending_up, 'growth_pct']:.2f}%)")
    print(f"Biggest Decline Category: {trending_down} ({valid_growth.loc[trending_down, 'growth_pct']:.2f}%)")
else:
    print("\nInsufficient data for growth trend analysis.")


--- Monthly Spending Matrix ---
month                 January    February        March       April  \
category                                                             
Cafe                 577.4470    882.0000   1707.67000   2665.2510   
Cash Withdrawal        0.2000      0.0000      0.00000    500.0000   
E-commerce         30664.3378  59258.4458  17978.44443  18112.1626   
Entertainment       1057.0000   2794.0000   1439.41700   2188.4880   
Food Delivery       7116.0350  10930.2960  13614.33400  11746.8110   
Fuel                8901.0000   1452.0000   3164.10505   8915.7480   
Groceries           1620.1258   1517.0000   2430.18650   1518.0000   
Investments            0.2720    821.0000  19141.00000    700.0000   
Miscellaneous       7760.0000  11915.0000    433.00000    683.8730   
Personal Transfer  44877.7842  21266.4212  42139.47871  64160.1387   
Quick Commerce      4272.4010   5807.0110   4059.30100   7321.2290   
Restaurants         4864.4640   6583.0000  18320.15110   3

/tmp/ipykernel_405/3259195199.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  month_pivot = df.pivot_table(


In [105]:
#Feature 6-Time-of-Day Patterns
import pandas as pd
import numpy as np

# 1. Clean Time Data: Convert to string and extract hour safely
# This handles potential NaNs or bad formatting before conversion
df['Time_cleaned'] = df['Time'].astype(str).str.strip()
df['hour'] = pd.to_numeric(df['Time_cleaned'].str[:2], errors='coerce')

# 2. Drop rows where hour couldn't be parsed to prevent the '00' column spike
# This ensures only valid 0-23 hours are mapped
df_clean = df.dropna(subset=['hour']).copy()
df_clean['hour'] = df_clean['hour'].astype(int)

# 3. Build the Matrix
# Use crosstab for category vs. hour frequency
time_matrix = pd.crosstab(df_clean['category'], df_clean['hour'])

# Ensure all 24 hours (0-23) exist in the matrix, even if spending is 0
time_matrix = time_matrix.reindex(columns=range(24), fill_value=0)

# 4. Generate the Visualization
def print_time_heatmap(matrix):
    print("\n--- Activity Heatmap (Hr 00-23) ---")

    # Header Row
    header = "Category    | " + " ".join([f"{h:02d}" for h in matrix.columns])
    print(header)
    print("-" * len(header))

    # Calculate normalization factor based on the data present
    max_val = matrix.values.max() if matrix.values.max() > 0 else 1

    # Build rows
    for category, row in matrix.iterrows():
        line = f"{str(category)[:10]:10} | "
        for val in row:
            intensity = val / max_val
            # Aesthetic blocks to match professional dashboard style
            if intensity == 0:   char = ' '   # Empty
            elif intensity < 0.25: char = '░'   # Light
            elif intensity < 0.50: char = '▒'   # Medium
            elif intensity < 0.75: char = '▓'   # Dark
            else:                 char = '█'   # Solid
            line += f" {char} "
        print(line)

# Run the visualization
print_time_heatmap(time_matrix)


--- Activity Heatmap (Hr 00-23) ---
Category    | 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
-------------------------------------------------------------------------------------
Cafe       |                                ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░        ░ 
Cash Withd |                                ░  ░        ░     ░     ░     ░  ░  ░    
E-commerce |                                ░  ▒  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░ 
Entertainm |                                                  ░     ░  ░  ░          
Food Deliv |                                ░  ▓  ▓  ▒  ░  ▒  ░  ▒  ▓  █  █  ▓  ▓  ░ 
Fuel       |                                   ░  ░  ░  ░  ░  ░        ░     ░     ░ 
Groceries  |                                ░  ░     ░  ░              ░  ░  ░       
Investment |                                ░        ░     ░  ░  ░        ░        ░ 
Miscellane |                                ░  ░  ░  ░  ░  ░  ░     ░     ░  ░     ░ 
Personal T |     

In [106]:
#Feature7 - Anomaly Detection
import numpy as np

#Compute category-specific statistics
df['cat_mean'] = df.groupby('category')['Amount_clean'].transform('mean')
df['cat_std'] = df.groupby('category')['Amount_clean'].transform('std')

#Compute Z-Score
#Formula:(Amount - Mean) / Standard Deviation
df['z_score'] = (df['Amount_clean'] - df['cat_mean']) / df['cat_std'].replace(0, np.nan)

#Identifying anomalies (Z-score > 2)
anomalies = df[df['z_score'] > 2].sort_values(by='z_score', ascending=False)
print("--- Top 5 Anomalies by Z-Score ---")
cols_to_display = ['Date', 'vendor_clean', 'category', 'Amount_clean', 'z_score']
print(anomalies[cols_to_display].head(5).to_string(index=False))

print(f"\nTotal anomalies detected: {len(anomalies)}")

--- Top 5 Anomalies by Z-Score ---
      Date vendor_clean          category  Amount_clean  z_score
2024-06-01 P2P Transfer Personal Transfer       85492.0 5.664346
2024-05-01 P2P Transfer Personal Transfer       85393.0 5.657425
       NaT P2P Transfer Personal Transfer       84736.0 5.611490
       NaT P2P Transfer Personal Transfer       84728.0 5.610931
       NaT P2P Transfer Personal Transfer       84724.0 5.610651

Total anomalies detected: 33


In [107]:
#Feature 8- Spending Archetype Detection
# Compute metrics based on categories and time
food_spend = df[df['category'] == 'Food Delivery']['Amount_clean'].sum()
invest_spend = df[df['category'] == 'Investments']['Amount_clean'].sum()
late_night_mask = (df['hour'].isin([23, 0, 1, 2, 3, 4])) & (df['category'].isin(['Food Delivery', 'Quick Commerce']))
late_night_spend = df[late_night_mask]['Amount_clean'].sum()
ecommerce_spend = df[df['category'] == 'E-commerce']['Amount_clean'].sum()
social_spend = df[df['category'].isin(['Restaurants', 'Cafe'])]['Amount_clean'].sum()

#assign archetypes
archetypes = []
if food_spend > 50000:
    archetypes.append(f"The Foodie: ₹{food_spend:,.2f} spent on food delivery.")
if invest_spend > 10000:
    archetypes.append(f"The Investor: ₹{invest_spend:,.2f} invested.")
if late_night_spend > 5000:
    archetypes.append(f"The Late-Night Snacker: ₹{late_night_spend:,.2f} spent on late-night cravings.")
if ecommerce_spend > 100000:
    archetypes.append(f"The E-commerce Addict: ₹{ecommerce_spend:,.2f} spent on online shopping.")
if social_spend > 30000:
    archetypes.append(f"The Social Butterfly: ₹{social_spend:,.2f} spent on dining out and cafes.")

print("--- Spending Archetypes Detected ---")
for arch in archetypes:
    print(f"• {arch}")

--- Spending Archetypes Detected ---
• The Foodie: ₹110,864.01 spent on food delivery.
• The Investor: ₹45,290.42 invested.
• The E-commerce Addict: ₹377,119.35 spent on online shopping.
• The Social Butterfly: ₹82,204.50 spent on dining out and cafes.


In [108]:
#The Spending Archetype System
#AI-assisted
# 1. Initialize 'is_debit' column
# We look for 'DR' or 'Debit' in the 'Type' column
if 'Type' in df.columns:
    df['is_debit'] = df['Type'].astype(str).str.contains('DR|Debit', case=False, na=False)
else:
    # If no 'Type' column exists, default all as debits for the analysis
    df['is_debit'] = True

# 2. Define Total Metrics
total_debits = df[df['is_debit']]['Amount_clean'].sum()
total_credits = df[~df['is_debit']]['Amount_clean'].sum()

# Helper function to get % of debits for a list of categories
def get_pct(cat_list):
    # Avoid division by zero
    if total_debits == 0: return 0
    return df[(df['is_debit']) & (df['category'].isin(cat_list))]['Amount_clean'].sum() / total_debits

archetypes = []

# 3. Apply Standard Rules from your Brief
if get_pct(['Food Delivery', 'Restaurants', 'Cafe']) > 0.25:
    archetypes.append("The Foodie")
if get_pct(['Quick Commerce']) > 0.15:
    archetypes.append("The Quick Commerce Junkie")
if get_pct(['E-commerce']) > 0.15:
    archetypes.append("The Shopaholic")
if get_pct(['Investments']) > 0.15:
    archetypes.append("The Investor")

# Late-night check: > 50% of Food Delivery transactions occur between 21:00 and 02:00
fd_df = df[(df['category'] == 'Food Delivery')]
late_night_count = len(fd_df[fd_df['hour'].isin([21, 22, 23, 0, 1, 2])])
if len(fd_df) > 0 and (late_night_count / len(fd_df)) > 0.5:
    archetypes.append("The Late-Night Snacker")

if get_pct(['Transport']) > 0.10:
    archetypes.append("The Cab Commuter")

# Subscription check: 5 or more distinct subscription vendors
sub_count = df[df['category'] == 'Subscriptions']['vendor_clean'].nunique()
if sub_count >= 5:
    archetypes.append("The Subscription Lover")

# Financial health check: YOLO vs Disciplined
savings_rate = (total_credits - total_debits) / total_credits if total_credits > 0 else -1
if savings_rate < 0.10:
    archetypes.append("The YOLO Spender")
elif savings_rate > 0.40:
    archetypes.append("The Disciplined Saver")

# Bonus: The Midnight Coder
midnight_mask = (df['hour'].isin([0, 1, 2, 3, 4])) & (df['category'].isin(['Quick Commerce', 'Cafe']))
if df[midnight_mask]['Amount_clean'].sum() > 2000:
    archetypes.append("The Midnight Coder (Bonus Archetype)")

# 4. Print Results
print("--- Spending Archetypes Detected ---")
for arch in archetypes:
    print(f"• {arch}")

--- Spending Archetypes Detected ---
• The Shopaholic
• The YOLO Spender
